# ============================================================
# MODEL DIAGNOSTICS — TUMC (134 features, binary + 5-class)
# ============================================================
# Modernized from the old binary/53-feature version:
#   - reads features_full_final_inductive.csv (honest set) if present,
#     else features_full_final.csv
#   - uses the C variant (drops is_tr_domain + is_https leakage)
#   - drops the 2 leaky graph features
#   - supports BOTH binary (label_enc) and 5-class (class_enc)
#   - 6-model roster matching Step 3 (no MLP/SVM/ExtraTrees/Cat/BalRF)
#
# 8 diagnostics:
#   1 Train/test gap (overfit)     5 Noise robustness
#   2 Learning curves              6 Feature quality/variance
#   3 Validation curves            7 Cross-fold variance
#   4 LOF outliers + PCA           8 Class balance + label noise
# ============================================================


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
SEED = 42
plt.rcParams.update({"font.size":11,"figure.dpi":150,
    "axes.spines.top":False,"axes.spines.right":False})

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
for cand in ["features_full_final_inductive_dedup.csv","features_full_final.csv"]:
    INPUT_FILE = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT_FILE): break
OUT_DIR = os.path.join(DATA_DIR, "diagnostic_plots")
os.makedirs(OUT_DIR, exist_ok=True)

TASK = "5class"     # "binary" or "5class" — run once each
GREEN,ORANGE,RED,BLUE = "#1D9E75","#E8762C","#C94F0B","#3A4A9E"
LEAKY = ["cluster_malicious_ratio","campaign_token_score","is_tr_domain","is_https"]

print("="*60)
print(f"MODEL DIAGNOSTICS — TUMC ({TASK})")
print(f"  Source: {os.path.basename(INPUT_FILE)}")
print("="*60)

df = pd.read_csv(INPUT_FILE, low_memory=False)
META = {"url","source","tld","label","label_enc","class_final",
        "class_enc","fold","reg_domain"}
FEATURES = [c for c in df.columns if c not in META and c not in LEAKY]
target = "label_enc" if TASK=="binary" else "class_enc"
is_binary = (TASK=="binary")
n_classes = df[target].nunique()

imp = SimpleImputer(strategy="median")
X = imp.fit_transform(df[FEATURES].values.astype(float))
y = df[target].values
folds = df["fold"].values
print(f"  X: {X.shape}  Features: {len(FEATURES)}  Classes: {n_classes}")

def auc_metric(y_true, proba):
    if is_binary:
        return roc_auc_score(y_true, proba[:,1])
    return roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
def f1_metric(y_true, pred):
    return f1_score(y_true, pred, average="binary" if is_binary else "macro", zero_division=0)

def make_models():
    M = {
        "LogisticRegression": LogisticRegression(C=1.0,max_iter=1000,
            class_weight="balanced",random_state=SEED,n_jobs=-1),
        "RandomForest": RandomForestClassifier(n_estimators=200,max_depth=12,
            min_samples_leaf=5,class_weight="balanced",random_state=SEED,n_jobs=-1),
        "HistGB": HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,
            max_iter=300,random_state=SEED),
        "XGBoost": xgb.XGBClassifier(max_depth=4,learning_rate=0.03,n_estimators=300,
            subsample=0.8,colsample_bytree=0.8,reg_alpha=1,reg_lambda=2,
            random_state=SEED,n_jobs=-1,verbosity=0,
            **({"scale_pos_weight":5.5} if is_binary else
               {"objective":"multi:softprob","num_class":n_classes})),
        "LightGBM": lgb.LGBMClassifier(num_leaves=31,max_depth=6,learning_rate=0.03,
            n_estimators=300,class_weight="balanced",random_state=SEED,
            n_jobs=-1,verbose=-1,
            **({} if is_binary else {"objective":"multiclass","num_class":n_classes})),
    }
    return M

# ── DIAG 1: train/test gap ───────────────────────────────────
print("\n[1] Train/test gap ...")
rows=[]
for name,model in make_models().items():
    tr_a,te_a=[],[]
    for fid in range(5):
        tri=np.where(folds!=fid)[0]; tei=np.where(folds==fid)[0]
        model.fit(X[tri],y[tri])
        tr_a.append(auc_metric(y[tri],model.predict_proba(X[tri])))
        te_a.append(auc_metric(y[tei],model.predict_proba(X[tei])))
    gap=np.mean(tr_a)-np.mean(te_a)
    v=("Healthy" if gap<0.01 else "Mild overfit" if gap<0.03 else "Overfit")
    if np.mean(te_a)<0.80: v="Underfit/hard"
    rows.append({"Model":name,"Train":round(np.mean(tr_a),4),
                 "Test":round(np.mean(te_a),4),"Gap":round(gap,4),"V":v})
    print(f"  {name:<18s}: train={np.mean(tr_a):.4f} test={np.mean(te_a):.4f} gap={gap:+.4f} {v}")
gdf=pd.DataFrame(rows)
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5)); xx=np.arange(len(gdf)); w=0.35
a1.bar(xx-w/2,gdf["Train"],w,label="Train",color=GREEN,alpha=0.8)
a1.bar(xx+w/2,gdf["Test"],w,label="Test",color=ORANGE,alpha=0.8)
a1.set_xticks(xx); a1.set_xticklabels(gdf["Model"],rotation=25,ha="right",fontsize=9)
a1.set_ylabel("AUC"); a1.legend(); a1.set_title("Train vs Test AUC",fontweight="bold")
gv=gdf["Gap"].values
a2.bar(gdf["Model"],gv,color=[GREEN if g<0.01 else ORANGE if g<0.03 else RED for g in gv],alpha=0.85)
a2.axhline(0.01,color=ORANGE,ls="--"); a2.axhline(0.03,color=RED,ls="--")
a2.set_xticklabels(gdf["Model"],rotation=25,ha="right",fontsize=9)
a2.set_ylabel("Train−Test gap"); a2.set_title("Overfitting gap",fontweight="bold")
plt.suptitle(f"Overfitting Diagnostic — TUMC ({TASK})",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/diag1_gap_{TASK}.png",bbox_inches="tight"); plt.close()

# ── DIAG 2: learning curves ──────────────────────────────────
print("[2] Learning curves ...")
si=np.random.RandomState(SEED).choice(len(X),min(20000,len(X)),replace=False)
Xs,ys=X[si],y[si]
lcm={"LogisticRegression":LogisticRegression(C=1.0,max_iter=500,random_state=SEED),
     "RandomForest":RandomForestClassifier(n_estimators=100,max_depth=12,random_state=SEED,n_jobs=-1),
     "XGBoost":xgb.XGBClassifier(n_estimators=100,max_depth=4,random_state=SEED,verbosity=0,n_jobs=-1,
        **({} if is_binary else {"objective":"multi:softprob","num_class":n_classes}))}
scoring="roc_auc" if is_binary else "f1_macro"
fig,axes=plt.subplots(1,3,figsize=(16,5))
for ax,(nm,mdl) in zip(axes,lcm.items()):
    ts,trs,vs=learning_curve(mdl,Xs,ys,train_sizes=np.linspace(0.1,1,8),
        cv=3,scoring=scoring,n_jobs=-1,shuffle=True,random_state=SEED)
    trm,vm=trs.mean(1),vs.mean(1)
    ax.plot(ts,trm,"o-",color=GREEN,label="Train"); ax.plot(ts,vm,"s--",color=ORANGE,label="Val")
    ax.fill_between(ts,trm-trs.std(1),trm+trs.std(1),alpha=0.15,color=GREEN)
    ax.fill_between(ts,vm-vs.std(1),vm+vs.std(1),alpha=0.15,color=ORANGE)
    g=trm[-1]-vm[-1]
    ax.set_title(f"{nm}\ngap={g:+.4f}",fontweight="bold",fontsize=10)
    ax.set_xlabel("Train size"); ax.set_ylabel(scoring); ax.legend(fontsize=9); ax.grid(alpha=0.2)
plt.suptitle(f"Learning Curves — TUMC ({TASK})",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/diag2_learning_{TASK}.png",bbox_inches="tight"); plt.close()

# ── DIAG 3: validation curves ────────────────────────────────
print("[3] Validation curves ...")
fig,axes=plt.subplots(1,2,figsize=(13,5))
cfgs=[(xgb.XGBClassifier(max_depth=4,random_state=SEED,verbosity=0,n_jobs=-1,
        **({} if is_binary else {"objective":"multi:softprob","num_class":n_classes})),
       "n_estimators",[25,50,100,150,200],"XGBoost — n_estimators"),
      (xgb.XGBClassifier(n_estimators=100,random_state=SEED,verbosity=0,n_jobs=-1,
        **({} if is_binary else {"objective":"multi:softprob","num_class":n_classes})),
       "max_depth",[2,3,4,5,6,8],"XGBoost — max_depth")]
for ax,(mdl,pn,pr,ti) in zip(axes,cfgs):
    trs,vs=validation_curve(mdl,Xs,ys,param_name=pn,param_range=pr,cv=3,scoring=scoring,n_jobs=-1)
    ax.plot(pr,trs.mean(1),"o-",color=GREEN,label="Train")
    ax.plot(pr,vs.mean(1),"s--",color=ORANGE,label="Val")
    bi=int(np.argmax(vs.mean(1)))
    ax.axvline(pr[bi],color=RED,ls=":",label=f"Best={pr[bi]}")
    ax.set_title(ti,fontweight="bold",fontsize=10); ax.set_xlabel(pn); ax.set_ylabel(scoring)
    ax.legend(fontsize=9); ax.grid(alpha=0.2)
plt.suptitle(f"Validation Curves — TUMC ({TASK})",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/diag3_validation_{TASK}.png",bbox_inches="tight"); plt.close()

# ── DIAG 4: LOF outliers + PCA ───────────────────────────────
print("[4] LOF outliers + PCA ...")
ns=5000; idx=np.random.RandomState(SEED).choice(len(X),ns,replace=False)
Xsa,ysa=X[idx],y[idx]; Xsc=StandardScaler().fit_transform(Xsa)
lof=LocalOutlierFactor(n_neighbors=20,contamination=0.03,n_jobs=-1)
ol=lof.fit_predict(Xsc); ls=-lof.negative_outlier_factor_
om=ol==-1; im=ol==1; no=om.sum()
Xp=PCA(n_components=2,random_state=SEED).fit_transform(Xsc)
fig=plt.figure(figsize=(18,5.5)); gs=gridspec.GridSpec(1,3,wspace=0.32)
ax1=fig.add_subplot(gs[0])
if is_binary:
    for cls,nm,col in [(0,"Benign",GREEN),(1,"Malicious",ORANGE)]:
        m=ysa==cls; ax1.scatter(Xp[m,0],Xp[m,1],c=col,alpha=0.25,s=8,label=nm,rasterized=True)
else:
    cmap=plt.cm.tab10
    for cls in range(n_classes):
        m=ysa==cls; ax1.scatter(Xp[m,0],Xp[m,1],alpha=0.3,s=8,label=f"class {cls}",rasterized=True)
ax1.set_title("PCA — Class Distribution",fontweight="bold"); ax1.legend(fontsize=8,markerscale=2); ax1.grid(alpha=0.2)
ax2=fig.add_subplot(gs[1])
ax2.scatter(Xp[im,0],Xp[im,1],c="steelblue",alpha=0.2,s=6,label="Inlier",rasterized=True)
ax2.scatter(Xp[om,0],Xp[om,1],c=RED,alpha=0.9,s=30,label=f"Outlier (n={no})",zorder=5)
ax2.set_title(f"LOF Outliers ({no/ns*100:.1f}%)",fontweight="bold"); ax2.legend(fontsize=9); ax2.grid(alpha=0.2)
ax3=fig.add_subplot(gs[2])
p995=np.percentile(ls,99.5)
lb=np.logspace(np.log10(max(ls.min(),1.0)),np.log10(max(p995,1.1)),40)
ax3.hist(np.clip(ls[im],1,p995),bins=lb,alpha=0.7,color="steelblue",density=True,label="Inlier")
ax3.hist(np.clip(ls[om],1,p995),bins=lb,alpha=0.8,color=ORANGE,density=True,label="Outlier")
ax3.axvline(np.percentile(ls,97),color="black",ls="--",label="97th pct")
ax3.set_xscale("log"); ax3.set_xlabel("LOF score (log)"); ax3.set_ylabel("Density")
ax3.set_title("LOF Score Distribution",fontweight="bold"); ax3.legend(fontsize=9); ax3.grid(axis="y",alpha=0.2)
plt.suptitle(f"Outlier Detection — TUMC ({TASK})",fontweight="bold",y=1.02)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/diag4_outliers_{TASK}.png",bbox_inches="tight"); plt.close()
print(f"  Outliers: {no}/{ns} ({no/ns*100:.1f}%)")

# ── DIAG 5: noise robustness ─────────────────────────────────
print("[5] Noise robustness ...")
nm_models={"LogisticRegression":LogisticRegression(C=1.0,max_iter=500,random_state=SEED),
   "RandomForest":RandomForestClassifier(n_estimators=100,max_depth=12,random_state=SEED,n_jobs=-1),
   "XGBoost":xgb.XGBClassifier(n_estimators=100,max_depth=4,random_state=SEED,verbosity=0,
      **({} if is_binary else {"objective":"multi:softprob","num_class":n_classes}))}
nlv=[0.0,0.01,0.02,0.05,0.10,0.15,0.20]
tri=np.where(folds!=0)[0]; tei=np.where(folds==0)[0]
nr={m:[] for m in nm_models}
for nz in nlv:
    Xn=X[tri].copy()
    if nz>0: Xn=Xn+np.random.RandomState(SEED).normal(0,nz*X[tri].std(0),X[tri].shape)
    for nm,mdl in nm_models.items():
        mdl.fit(Xn,y[tri]); nr[nm].append(auc_metric(y[tei],mdl.predict_proba(X[tei])))
fig,ax=plt.subplots(figsize=(9,6))
for (nm,a),col in zip(nr.items(),[GREEN,ORANGE,BLUE]):
    ax.plot([n*100 for n in nlv],a,"o-",color=col,lw=2,label=f"{nm} (drop={a[0]-a[-1]:+.4f})")
ax.set_xlabel("Train noise (%)"); ax.set_ylabel("Test AUC")
ax.set_title(f"Noise Robustness — TUMC ({TASK})",fontweight="bold"); ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/diag5_noise_{TASK}.png",bbox_inches="tight"); plt.close()

# ── DIAG 6: feature quality ──────────────────────────────────
print("[6] Feature quality ...")
fdf=pd.DataFrame(X,columns=FEATURES); var=fdf.var()
nz=var[var<0.001]
lc=fdf.corrwith(pd.Series(y)).abs().sort_values(ascending=False)
fig,axes=plt.subplots(1,2,figsize=(15,6))
lc.head(20).plot(kind="barh",ax=axes[0],color=ORANGE,alpha=0.8)
axes[0].set_title("Top 20 |corr| with label",fontweight="bold"); axes[0].invert_yaxis()
if len(nz)>0:
    nz.sort_values().plot(kind="barh",ax=axes[1],color=RED,alpha=0.8)
    axes[1].set_title(f"Near-zero variance (n={len(nz)})",fontweight="bold")
else:
    axes[1].text(0.5,0.5,"No near-zero variance features",ha="center",va="center",
        color=GREEN,fontsize=13,fontweight="bold",transform=axes[1].transAxes); axes[1].axis("off")
plt.suptitle(f"Feature Quality — TUMC ({TASK})",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/diag6_features_{TASK}.png",bbox_inches="tight"); plt.close()
print(f"  Near-zero var: {len(nz)}")

# ── DIAG 7: cross-fold variance ──────────────────────────────
print("[7] Cross-fold variance ...")
cv=[]
for nm,mdl in make_models().items():
    fa=[]
    for fid in range(5):
        tri=np.where(folds!=fid)[0]; tei=np.where(folds==fid)[0]
        mdl.fit(X[tri],y[tri]); fa.append(auc_metric(y[tei],mdl.predict_proba(X[tei])))
    cv.append({"Model":nm,"aucs":fa,"mean":np.mean(fa),"std":np.std(fa)})
cvd=pd.DataFrame(cv)
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,6))
cols=[GREEN,"#0F6E56",ORANGE,RED,BLUE,"#3A4A9E"]
for i,r in cvd.iterrows():
    a1.plot(range(5),r["aucs"],"o-",color=cols[i%len(cols)],lw=2,label=f'{r["Model"]} (s={r["std"]:.4f})')
a1.set_xlabel("Fold"); a1.set_ylabel("AUC"); a1.set_title("AUC per fold",fontweight="bold")
a1.legend(fontsize=8.5); a1.grid(alpha=0.25); a1.set_xticks(range(5))
cvs=cvd.sort_values("std")
a2.barh(cvs["Model"],cvs["std"],color=[GREEN if s<0.005 else ORANGE if s<0.01 else RED for s in cvs["std"]],alpha=0.85)
a2.set_xlabel("AUC std"); a2.set_title("Stability (lower=better)",fontweight="bold"); a2.grid(axis="x",alpha=0.25)
plt.suptitle(f"Cross-Fold Stability — TUMC ({TASK})",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/diag7_crossfold_{TASK}.png",bbox_inches="tight"); plt.close()

# ── DIAG 8: class balance + label noise ──────────────────────
print("[8] Class balance + label noise ...")
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
if is_binary:
    fb,fm=[],[]
    for fid in range(5):
        ty=y[np.where(folds==fid)[0]]; fb.append((ty==0).sum()); fm.append((ty==1).sum())
    xp=np.arange(5); w=0.35
    ax.bar(xp-w/2,fb,w,label="Benign",color=GREEN,alpha=0.8)
    ax.bar(xp+w/2,fm,w,label="Malicious",color=ORANGE,alpha=0.8)
    ax.set_xticks(xp); ax.set_xticklabels([f"F{i}" for i in range(5)]); ax.legend()
else:
    comp=df.groupby(["fold","class_final"]).size().unstack(fill_value=0)
    comp.plot(kind="bar",stacked=True,ax=ax,colormap="tab10")
    ax.legend(fontsize=7)
ax.set_title("Class balance per fold",fontweight="bold"); ax.set_ylabel("Count")
# label noise via high-conf-wrong (binary only meaningful)
ax=axes[1]
et=xgb.XGBClassifier(n_estimators=100,max_depth=4,random_state=SEED,verbosity=0,n_jobs=-1,
    **({} if is_binary else {"objective":"multi:softprob","num_class":n_classes}))
at,ap=[],[]
for fid in range(5):
    tri=np.where(folds!=fid)[0]; tei=np.where(folds==fid)[0]
    et.fit(X[tri],y[tri]); pr=et.predict_proba(X[tei])
    at.extend(y[tei]); ap.append(pr)
ap=np.vstack(ap); at=np.array(at)
if is_binary:
    p1=ap[:,1]
    hcw=((p1>0.95)&(at==0))|((p1<0.05)&(at==1))
    ns_=hcw.sum()
    ax.hist(p1[at==0],bins=50,alpha=0.65,color=GREEN,density=True,label="Benign")
    ax.hist(p1[at==1],bins=50,alpha=0.65,color=ORANGE,density=True,label="Malicious")
    ax.axvline(0.05,color=RED,ls=":"); ax.axvline(0.95,color=RED,ls=":")
    ax.set_xlabel("P(malicious)"); ax.set_title(f"Label noise: {ns_} hi-conf-wrong ({ns_/len(at)*100:.2f}%)",fontweight="bold")
    ax.legend(fontsize=9)
else:
    conf=ap.max(1); pred=ap.argmax(1)
    hcw=(conf>0.95)&(pred!=at); ns_=hcw.sum()
    ax.hist(conf[pred==at],bins=50,alpha=0.65,color=GREEN,density=True,label="Correct")
    ax.hist(conf[pred!=at],bins=50,alpha=0.65,color=RED,density=True,label="Wrong")
    ax.set_xlabel("Max class probability"); ax.legend(fontsize=9)
    ax.set_title(f"Label noise: {ns_} hi-conf-wrong ({ns_/len(at)*100:.2f}%)",fontweight="bold")
plt.suptitle(f"Data Quality — TUMC ({TASK})",fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/diag8_quality_{TASK}.png",bbox_inches="tight"); plt.close()
print(f"  Hi-conf-wrong: {ns_} ({ns_/len(at)*100:.2f}%)")

print(f"\n{'='*60}\nDIAGNOSTICS COMPLETE ({TASK}) → {OUT_DIR}/\n{'='*60}")
print("  Set TASK='5class' and rerun for multi-class diagnostics.")

MODEL DIAGNOSTICS — TUMC (5class)
  Source: features_full_final_inductive_dedup.csv
  X: (1239308, 135)  Features: 135  Classes: 5

[1] Train/test gap ...
  LogisticRegression: train=0.7210 test=0.7191 gap=+0.0018 Underfit/hard
  RandomForest      : train=0.9766 test=0.9696 gap=+0.0070 Healthy
  HistGB            : train=0.9881 test=0.9811 gap=+0.0070 Healthy
  XGBoost           : train=0.9767 test=0.9729 gap=+0.0039 Healthy
  LightGBM          : train=0.9836 test=0.9780 gap=+0.0057 Healthy
[2] Learning curves ...
[3] Validation curves ...
[4] LOF outliers + PCA ...
  Outliers: 150/5000 (3.0%)
[5] Noise robustness ...


KeyboardInterrupt: 